<div style="
background-color:#EAEAEA;
padding:20px;
border-left:5px solid #6C757D;
border-radius:6px;">

<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="border:none; vertical-align:top;">

<h1 style="font-size:32px; margin-top:0;">
Master's Thesis
</h1>

<hr style="margin:16px 0 22px 0;">

<p style="font-size:22px; line-height:1.5; margin:0;">
<strong>Master's Degree in Advanced Physics</strong> - 
<strong>Universitat de Val?ncia</strong>
</p>

<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">
This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:
</p>

<div style="
font-size:25px;
font-weight:700;
line-height:1.3;
margin-top:14px;
margin-bottom:26px;">
Fast Simulation of Neutrino Oscillations in Matter
</div>

<p style="font-size:14px; line-height:1.55;">
<strong>Author</strong><br>
Juan Ramon Diaz Santos - 
<a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a>
</p>

<p style="font-size:14px; line-height:1.55;">
<strong>Supervisors</strong><br>
Roberto Ruiz de Austri Bazan ?
<a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>
Michele Lucente ?
<a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a>
</p>

<p style="font-size:14px; line-height:1.55; margin-bottom:0;">
<strong>Date</strong><br>
September 2026
</p>

</td>

<td style="
border:none;
width:230px;
padding-left:25px;
text-align:right;
vertical-align:top;">

<img src="../../logo_uv.png"
     alt="Universitat de Val?ncia"
     style="width:200px; margin-top:5px;">

</td>

</tr>
</table>

</div>

# Intrinsic Validation 2 — BSM (NSI / Sterile) Perturbative versus Numerical Propagation

This notebook extends [Intrinsic Validation 1](../intrinsic/validation_intrinsic1_perturvative_vs_numerical.ipynb) to the two BSM extensions implemented in TPeanuts: Non-Standard Interactions (NSI) and the 3+1 sterile-neutrino scheme. It checks two independent things at once for every scenario below:

1. **BSM correctness**: the NSI and sterile Hamiltonians reduce to the Standard Model exactly at their trivial limit (`epsilon -> 0`, sterile mixing angles `-> 0`), for both propagation methods independently.
2. **Perturbative/numerical agreement**: the first-order perturbative (`analytical`) evolutor and the segmented matrix-exponential (`numerical`) evolutor agree with each other across a representative set of literature NSI and sterile benchmark presets, for both solar and atmospheric neutrinos reaching the detector.

---

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: NSI and sterile Hamiltonians, validation methodology |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Solar-Neutrinos-to-the-Detector:-SM-versus-BSM-Limits) | **Solar Neutrinos to the Detector: SM versus BSM Limits** |
| [4](#4.-NSI-Presets----Solar-Neutrinos-to-the-Detector) | **NSI Presets — Solar Neutrinos to the Detector** |
| [5](#5.-NSI-Presets----Atmospheric-Neutrinos-to-the-Detector) | **NSI Presets — Atmospheric Neutrinos to the Detector** |
| [6](#6.-Sterile-Presets----Solar-Neutrinos-to-the-Detector) | **Sterile Presets — Solar Neutrinos to the Detector** |
| [7](#7.-Sterile-Presets----Atmospheric-Neutrinos-to-the-Detector) | **Sterile Presets — Atmospheric Neutrinos to the Detector** |
| [∑](#8.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 Non-Standard Interactions (NSI)

NSI are effective four-fermion operators that add a flavour-dependent piece to the coherent forward-scattering matter potential *(Grossman 1995; Biggio, Blennow & Fernandez-Martinez 2009)*:

$$H_{\rm mat}^{\rm NSI}=V_{CC}\,\bigl(\mathrm{diag}(1,0,0)+\varepsilon\bigr),\qquad V_{CC}=\pm\sqrt{2}\,G_F n_e,$$

where $\varepsilon$ is a Hermitian $3\times3$ dimensionless coupling matrix. $\varepsilon=0$ recovers the Standard Model exactly. TPeanuts builds this term with `tpeanuts.core.common.hamiltonian.hamiltonian_matter_reduced` and rotates it into whichever basis (reduced or flavour) the perturbative or numerical evolutor needs.

### 0.2 The 3+1 sterile extension

A fourth, weak-singlet mass eigenstate is added, extending the PMNS matrix to $4\times4$ and the kinetic Hamiltonian's mass-squared vector with a fourth entry $\Delta m^2_{41}$ *(Giunti, Marrone & Palazzo 2017)*. Because the sterile state carries no charged- or neutral-current coupling, the matter potential remains confined to the electron-flavour entry, $H_{\rm mat}=\mathrm{diag}(V_{CC},0,0,0)$. TPeanuts builds the $4\times4$ mixing matrix in `tpeanuts.core.BSM.bsm_sterile.PMNS_sterile` and generalizes every propagation path (perturbative segment, numerical segment, Earth mirroring) to work for $N\in\{3,4\}$ without a separate code path.

### 0.3 Two independent limits to the Standard Model

* **NSI limit**: $\varepsilon\to 0$. Because the analytical and numerical BSM branches both dispatch through a *different* code path than the pure-SM one (a full $\varepsilon$-aware Hamiltonian is still assembled and diagonalized/exponentiated even at $\varepsilon=0$), comparing $\varepsilon=0$ against the plain SM calculation is a genuine, non-trivial regression check, not a tautology.
* **Sterile limit**: all three active-sterile mixing angles $\theta_{14}=\theta_{24}=\theta_{34}=0$. The `sterile_3p1_null_mixing` preset builds a full $4\times4$ `PMNS_sterile` object with this limit; the fourth (sterile) flavour then carries zero production and zero propagation probability, and the first three flavours must reproduce the pure 3-flavour Standard Model to numerical precision.

### 0.4 Physical scenarios

Solar neutrinos are produced in the Sun as incoherent mass-eigenstate weights (the Sun-Earth baseline averages relative phases) and are then propagated through Earth matter to the detector: `solar_probability_mass` (production) followed by `earth_probability_state` (Earth crossing). For the sterile limit, the solar production step itself has no sterile generalization in TPeanuts (the sterile state is never produced by weak interactions in the solar core), so this notebook computes the incoherent weights with the *active* three-flavour sector of the preset and appends a zero sterile weight before propagating through the full $4\times4$ Earth evolutor -- the standard treatment for solar+sterile scenarios in the literature.

Atmospheric neutrinos remain coherent from production to detector, so the two-stage chain composes amplitudes, $S_{\rm total}=S_{\rm Earth}S_{\rm atm}$, rather than probabilities: `atmosphere_evolutor` builds $S_{\rm atm}$, and the resulting surface-level amplitude is propagated through `earth_probability_state` in flavour-amplitude mode. Both NSI and sterile presets are exercised for the initial $\nu_\mu$ flavour, the channel most commonly targeted by atmospheric NSI/sterile searches (DeepCore, IceCube).

### 0.5 Validation observables

Every comparison in this notebook reports, over its full energy/angle grid:

* $\max|\Delta P|$ and $\mathrm{median}|\Delta P|$, with $\Delta P=P_1-P_2$;
* $\max$ and $\mathrm{median}$ of $|\Delta P|/\max(|P_2|,10^{-12})$;
* the probability-conservation (unitarity) error $\max_\alpha|\sum_\beta P_{\alpha\to\beta}-1|$ for **both** compared tensors, reported as `unitarity_1` (first argument) and `unitarity_2` (second argument) -- which physical quantity is "1" and which is "2" is stated by each row's `case`/`check` fields (e.g. "analytical" vs "numerical" for the preset-comparison sections, or a BSM limit vs the SM baseline for Section 3).

Sections 3-7 each display a single combined table of every case computed in that section (rather than one table per case). Every row's `case` is reduced to the two configurations being compared, with the propagation chain and comparison type factored out into separate `scenario`/`check` columns (e.g. `scenario="Solar to detector"`, `check="numerical"` for Section 3; `scenario="Atmosphere to detector (nu_mu)"`, `check="analytical vs numerical"` for one named preset in Section 5).

**Reading `max_rel`:** relative error is a ratio, so it is intrinsically unstable near a probability zero-crossing or a strongly suppressed oscillation channel: if both compared probabilities are genuinely tiny (e.g. $\sim10^{-5}$) at one particular grid point, a perfectly ordinary $\sim10^{-5}$ absolute mismatch there divides out to an $O(1)$ relative error, even though the two methods agree everywhere else to high precision. This is a well-known artefact of relative-error metrics, not evidence of a propagation bug -- always cross-check a large `max_rel` against `max_abs` (is the absolute mismatch actually large, and at the *same* grid point?) and against `median_rel` (which stays tiny whenever the large `max_rel` is an isolated near-zero-crossing point rather than a systematic effect).

### References

- Y. Grossman, *Non-standard neutrino interactions and neutrino oscillation experiments*, Phys. Lett. B **359**, 141 (1995).
- C. Biggio, M. Blennow and E. Fernandez-Martinez, *General bounds on non-standard neutrino interactions*, JHEP **08**, 090 (2009), arXiv:0907.0097.
- E. Esteban, M. C. Gonzalez-Garcia, M. Maltoni, I. Martinez-Soler and T. Schwetz, *Updated constraints on non-standard interactions from global analysis of oscillation data*, JHEP **08**, 180 (2018), arXiv:1805.04530.
- IceCube Collaboration, *Search for non-standard neutrino interactions with IceCube DeepCore*, Phys. Rev. D **106**, 032009 (2022), arXiv:2112.09122.
- C. Giunti, M. Marrone and A. Palazzo, *Combined 3+1 global analysis*, arXiv:1612.01087 (2017).
- IceCube Collaboration, *eV-scale sterile neutrino search with IceCube*, PRL **125**, 141801 (2020), arXiv:2005.12943.

## 1. Libraries

In [ ]:
from __future__ import annotations

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import save_and_show, to_numpy
from tpeanuts.core.common.oscillation import OscillationParameters
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.core.common.pmns import PMNSParams
from tpeanuts.core.SM.sm_pmns import PMNS_SM
from tpeanuts.core.SM.sm_mass_spectrum import MassSpectrum_SM
from tpeanuts.core.BSM.bsm_nsi import NSIConfig
from tpeanuts.medium.earth.profile import EarthParameters, EarthProfile
from tpeanuts.medium.earth.probability import earth_probability_state
from tpeanuts.medium.solar.profile import SolarProfile
from tpeanuts.medium.solar.probability import solar_probability_mass
from tpeanuts.medium.atmosphere.profile import AtmosphereParameters
from tpeanuts.medium.atmosphere.evolutor import atmosphere_evolutor
from tpeanuts.util.context import RuntimeContext

## 2. Paths and Configuration

### 2.1 Paths

`load_notebook_config()` resolves the package directory and applies the common plotting, Torch and NumPy style. All generated figures and tables are saved under `validation/intrinsic/`.

**Expected results:** the package and output directories exist, and the runtime device and dtype match the shared notebook configuration.

In [ ]:
config = load_notebook_config()
DEVICE, DTYPE = config.device, config.dtype
CDTYPE = torch.complex128 if DTYPE == torch.float64 else torch.complex64
context = RuntimeContext.resolve(DEVICE, DTYPE)
OUTPUT_DIR = config.output_dir('validation', 'intrinsic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Package dir : {config.package_dir}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Device      : {context.device}   dtype: {context.dtype}')

### 2.2 Configuration

Earth uses the package PREM polynomial profile (matching Intrinsic Validation 1), with a genuinely varying (non-constant) density per shell -- important here because the NSI first-order perturbative correction depends on the residual density shape, unlike a constant-density segment. Solar $^8$B neutrinos use the default tabulated solar model. The atmosphere uses the exponential density source with a moderate perturbative fit (fewer segments/steps than Intrinsic Validation 1, traded for the much larger number of scenarios compared here).

| Parameter | Value | Description |
|-----------|-------|-------------|
| SM baseline preset | `_SM_NUFIT52_NO` | Three-flavour normal ordering |
| Sterile null-mixing preset | `sterile_3p1_null_mixing` | 3+1 machinery, all sterile angles at zero -> exact SM limit |
| NSI SM-limit | `sm_no_nsi` | All `epsilon` entries zero |
| NSI preset subset | 5 presets | See Section 4 |
| Sterile preset subset | 3 presets | See Section 6 |
| Earth profile | PREM even-power layers | Non-constant density (exercises the first-order correction) |
| Earth numerical steps | 250 | Midpoint constant-density segments |
| Atmosphere profile | Exponential | 6 cubic analytical segments; 300 numerical steps |
| Detector depth | 1000 m | Buried detector |
| Solar grid (Sections 3, 4, 6) | 25-35 x 25-35 | Energy x nadir angle |
| Atmosphere grid (Sections 5, 7) | 20 x 20 | Energy x zenith angle, initial $\nu_\mu$ only |
| Relative-difference floor | $10^{-12}$ | Stabilizes ratios close to zero |

**Expected results:** all profiles and presets load successfully; every probability tensor uses the configured device and real/complex precision.

In [ ]:
SM_PRESET = '_SM_NUFIT52_NO'
STERILE_NULL_PRESET = 'sterile_3p1_null_mixing'
NSI_SM_PRESET = 'sm_no_nsi'

NSI_PRESETS = [
    'nsi_ee_only',
    'nsi_dune_etau',
    'nsi_globalfit_esteban2018',
    'nsi_lma_dark_esteban2018',
    'nsi_offdiag_icecube2022',
]
STERILE_PRESETS = [
    'sterile_3p1_bestfit_giunti2017',
    'sterile_3p1_benchmark_icecube',
    'sterile_3p1_two_flavour_limit',
]

sm_oscillation = PropagationConfig.oscillation_parameters_from_preset(SM_PRESET, context=context, antinu=False)
sterile_null_oscillation = PropagationConfig.oscillation_parameters_from_preset(STERILE_NULL_PRESET, context=context, antinu=False)
eps_zero = NSIConfig.from_preset(NSI_SM_PRESET).epsilon_tensor(device=DEVICE, real_dtype=DTYPE)

earth = EarthProfile(
    params=EarthParameters(profile_perturbative_kwargs={'density_file': str(config.earth_density_file), 'tabulated_density': False}),
    context=context,
)
solar = SolarProfile.default(context=context)
atmosphere = AtmosphereParameters(atmosphere_density_source='exponential', nsteps=300, matter=True, perturbative_segments=6, perturbative_degree=3)

DEPTH_M, N_EARTH_STEPS, H_KM = 1000.0, 250, 25.0
FLAVOURS = ('nu_e', 'nu_mu', 'nu_tau', 'nu_s')

print(f'SM preset: {sm_oscillation.preset_name} ({sm_oscillation.ordering})')
print(f'Sterile null-mixing preset: {sterile_null_oscillation.preset_name}, n_flavours={sterile_null_oscillation.pmns.n_flavours}')
print(f'NSI presets: {NSI_PRESETS}')
print(f'Sterile presets: {STERILE_PRESETS}')
print(f'Earth steps={N_EARTH_STEPS}; atmosphere steps={atmosphere.nsteps}; analytical atmosphere={atmosphere.perturbative_segments}x degree {atmosphere.perturbative_degree}')

### 2.3 Validation helpers

`compute_metrics` computes the discrepancy and unitarity metrics shared by every section, without displaying anything. `make_row` wraps it into one table row (`case` plus any extra labelling fields, e.g. `scenario`/`check`) and appends it to the global `summaries` list collected in Section 8. `show_table` displays one combined, formatted table for a section's rows. `sm_sector_oscillation`, `solar_to_detector`, and `atmosphere_to_detector` implement the two propagation chains once, parametrized by method/epsilon/oscillation so every later section reuses them unchanged. `heatmap_pair` and `energy_line_triplet` draw, respectively, an Intrinsic-Validation-1-style independent absolute/relative error heatmap over the full grid, and a $\nu_e$-probability/absolute-error/relative-error triplet along an energy scan at fixed angle, both re-used once per preset in Sections 4-7. `bar_plot` draws the per-preset maximum absolute/relative error used to compare presets within a section at a glance.

**Expected results:** each scenario below appends one finite summary row; every figure is written into `OUTPUT_DIR`.

In [ ]:
REL_FLOOR = 1e-12
summaries = []

def compute_metrics(P1, P2):
    """Discrepancy and unitarity metrics between two probability tensors (any matching final flavour-count dim)."""
    absolute = torch.abs(P1 - P2)
    relative = absolute / torch.clamp(torch.abs(P2), min=REL_FLOOR)
    unit1 = torch.amax(torch.abs(P1.sum(dim=-1) - 1.0)).item()
    unit2 = torch.amax(torch.abs(P2.sum(dim=-1) - 1.0)).item()
    return {
        'max_abs': absolute.max().item(),
        'median_abs': absolute.median().item(),
        'max_rel': relative.max().item(),
        'median_rel': relative.median().item(),
        'unitarity_1': unit1,
        'unitarity_2': unit2,
    }


def make_row(case, P1, P2, **extra):
    """Build one summary-table row; which quantity is "1"/"2" is documented by ``case``/``extra``."""
    return {'case': case, **extra, **compute_metrics(P1, P2)}


def show_table(rows):
    df = pd.DataFrame(rows).set_index('case')
    numeric_cols = df.select_dtypes(include='number').columns
    display(df.style.format({c: '{:.3e}' for c in numeric_cols}))
    return df


def sm_sector_oscillation(sterile_oscillation):
    """Build the plain 3-flavour OscillationParameters sharing a sterile preset's SM-sector angles."""
    sm_pmns = PMNS_SM(sterile_oscillation.pmns.params)
    mass_spectrum = MassSpectrum_SM(
        DeltamSq21=sterile_oscillation.mass_spectrum.DeltamSq21,
        DeltamSq3l=sterile_oscillation.mass_spectrum.DeltamSq3l,
    )
    return OscillationParameters(
        pmns=sm_pmns, mass_spectrum=mass_spectrum, antinu=sterile_oscillation.antinu,
    )


def sterile_solar_weights(sterile_oscillation, E_grid):
    """Active-sector solar production weights, zero-padded for the sterile flavour."""
    sm_osc = sm_sector_oscillation(sterile_oscillation)
    weights3 = solar_probability_mass(sm_osc, E_grid, solar, '8B')
    zeros = torch.zeros_like(weights3[..., :1])
    return torch.cat([weights3, zeros], dim=-1)


def solar_to_detector(weights, oscillation, E_grid, eta_grid, method, *, epsilon=None):
    """Propagate precomputed solar mass-basis weights through Earth to the detector."""
    results = [
        earth_probability_state(
            weights, earth, oscillation, E_grid[:, None], eta, DEPTH_M,
            method=method, massbasis=True, nsteps=N_EARTH_STEPS, context=context,
            reunitarize=True, epsilon=epsilon,
        ).reshape(E_grid.numel(), -1)
        for eta in eta_grid
    ]
    return torch.stack(results, dim=1)


def atmosphere_to_detector(state, oscillation, E_grid, theta_grid, method, *, epsilon=None):
    """Coherently propagate one initial flavour state: production -> atmosphere -> Earth -> detector."""
    S_atm, _ = atmosphere_evolutor(
        oscillation, E_grid[:, None], H_KM, theta_grid[None, :], DEPTH_M / 1000.0,
        atmosphere=atmosphere, context=context, method=method, epsilon=epsilon,
    )
    results = []
    for i_theta in range(theta_grid.numel()):
        eta = torch.pi - torch.deg2rad(theta_grid[i_theta])
        surface_state = torch.matmul(S_atm[:, i_theta], state)
        P = earth_probability_state(
            surface_state, earth, oscillation, E_grid[:, None], eta, DEPTH_M,
            method=method, massbasis=False, nsteps=N_EARTH_STEPS, context=context,
            reunitarize=True, epsilon=epsilon,
        ).reshape(E_grid.numel(), -1)
        results.append(P)
    return torch.stack(results, dim=1)


def bar_plot(rows, title, filename):
    """2x2 grid: max/median absolute error (top) and max/median relative error (bottom), per case."""
    labels = [r['case'] for r in rows]
    panels = (
        ('max_abs', 'Maximum absolute error', 'C0'),
        ('max_rel', 'Maximum relative error', 'C1'),
        ('median_abs', 'Median absolute error', 'C2'),
        ('median_rel', 'Median relative error', 'C3'),
    )
    fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.6))
    for ax, (key, subtitle, colour) in zip(axes.flat, panels):
        ax.bar(range(len(labels)), [r[key] for r in rows], color=colour)
        ax.set_title(subtitle); ax.set_yscale('log')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
        ax.grid(alpha=0.25, axis='y')
    fig.suptitle(title); fig.tight_layout()
    save_and_show(filename, fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)


def heatmap_pair(P1, P2, x, y, title, xlabel, ylabel, filename):
    """Independent absolute/relative error heatmap over the full (x, y) grid, maxed over the flavour axis."""
    diff = torch.abs(P1 - P2)
    absolute = diff.amax(dim=-1)
    relative = (diff / torch.clamp(torch.abs(P2), min=REL_FLOOR)).amax(dim=-1)
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), sharex=True, sharey=True)
    for ax, values, subtitle, clabel in zip(
        axes, (absolute, relative), ('Absolute error', 'Relative error'),
        ('max |Delta P|', 'max |Delta P| / max(|P_2|, 1e-12)'),
    ):
        im = ax.pcolormesh(to_numpy(x), to_numpy(y), to_numpy(values).T, shading='auto', cmap='magma')
        ax.set(xlabel=xlabel, ylabel=ylabel, title=subtitle)
        fig.colorbar(im, ax=ax, label=clabel)
    fig.suptitle(title); fig.tight_layout()
    save_and_show(filename, fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)


def energy_line_triplet(P1, P2, E_grid, i_axis2, label1, label2, title, filename, *, log_energy=False, flavour_index=0, flavour_label='nu_e'):
    """Horizontal P(flavour)/absolute-error/relative-error triplet along an energy scan at a fixed second-axis index."""
    p1 = P1[:, i_axis2, flavour_index]
    p2 = P2[:, i_axis2, flavour_index]
    absolute = torch.abs(p1 - p2)
    relative = absolute / torch.clamp(torch.abs(p2), min=REL_FLOOR)
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))
    axes[0].plot(to_numpy(E_grid), to_numpy(p1), label=label1)
    axes[0].plot(to_numpy(E_grid), to_numpy(p2), '--', label=label2)
    axes[0].set_title(f'P({flavour_label})'); axes[0].set_ylabel('Probability'); axes[0].legend(fontsize=8)
    axes[1].plot(to_numpy(E_grid), to_numpy(absolute), color='C3')
    axes[1].set_title('Absolute error'); axes[1].set_ylabel('|Delta P|'); axes[1].set_yscale('log')
    axes[2].plot(to_numpy(E_grid), to_numpy(relative), color='C4')
    axes[2].set_title('Relative error'); axes[2].set_ylabel('|Delta P| / max(|P_2|, 1e-12)'); axes[2].set_yscale('log')
    for ax in axes:
        ax.set_xlabel('E [MeV]'); ax.grid(alpha=0.25)
        if log_energy:
            ax.set_xscale('log')
    fig.suptitle(title); fig.tight_layout()
    save_and_show(filename, fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)

## 3. Solar Neutrinos to the Detector: SM versus BSM Limits

Solar $^8$B neutrinos are propagated to the detector via three independent oscillation configurations that should agree numerically: the plain Standard Model, the NSI branch evaluated at $\varepsilon=0$ (`sm_no_nsi`), and the 3+1 sterile branch evaluated at null mixing (`sterile_3p1_null_mixing`, with the fourth flavour's production weight fixed at zero, see Section 0.4). Each BSM limit is compared against the SM baseline, first using the numerical propagation method for all three, then again using the analytical method for all three.

**Expected results:** every BSM-limit-vs-SM discrepancy is at the level of floating-point/perturbative-truncation noise, several orders of magnitude below any genuine BSM effect explored in the later sections; the sterile flavour's production+propagation probability stays at exactly zero.

In [ ]:
E_solar_lim = torch.linspace(0.5, 15.0, 35, device=DEVICE, dtype=DTYPE)
eta_solar_lim = torch.linspace(0.10, 1.45, 35, device=DEVICE, dtype=DTYPE)

weights_SM = solar_probability_mass(sm_oscillation, E_solar_lim, solar, '8B')
weights_NSI0 = solar_probability_mass(sm_oscillation, E_solar_lim, solar, '8B', epsilon=eps_zero)
weights_Sterile0 = sterile_solar_weights(sterile_null_oscillation, E_solar_lim)

section3_rows = []

P_SM_num = solar_to_detector(weights_SM, sm_oscillation, E_solar_lim, eta_solar_lim, 'numerical')
P_NSI0_num = solar_to_detector(weights_NSI0, sm_oscillation, E_solar_lim, eta_solar_lim, 'numerical', epsilon=eps_zero)
P_Sterile0_num = solar_to_detector(weights_Sterile0, sterile_null_oscillation, E_solar_lim, eta_solar_lim, 'numerical')

section3_rows.append(make_row('NSI-limit vs SM', P_NSI0_num, P_SM_num, scenario='Solar to detector', check='numerical'))
section3_rows.append(make_row('Sterile-limit vs SM', P_Sterile0_num[..., :3], P_SM_num, scenario='Solar to detector', check='numerical'))
print(f"Sterile-flavour leakage (numerical), max|P_s|: {P_Sterile0_num[..., 3].abs().max().item():.3e}")

In [ ]:
P_SM_ana = solar_to_detector(weights_SM, sm_oscillation, E_solar_lim, eta_solar_lim, 'analytical')
P_NSI0_ana = solar_to_detector(weights_NSI0, sm_oscillation, E_solar_lim, eta_solar_lim, 'analytical', epsilon=eps_zero)
P_Sterile0_ana = solar_to_detector(weights_Sterile0, sterile_null_oscillation, E_solar_lim, eta_solar_lim, 'analytical')

section3_rows.append(make_row('NSI-limit vs SM', P_NSI0_ana, P_SM_ana, scenario='Solar to detector', check='analytical'))
section3_rows.append(make_row('Sterile-limit vs SM', P_Sterile0_ana[..., :3], P_SM_ana, scenario='Solar to detector', check='analytical'))
print(f"Sterile-flavour leakage (analytical), max|P_s|: {P_Sterile0_ana[..., 3].abs().max().item():.3e}")

summaries.extend(section3_rows)
show_table(section3_rows)

In [ ]:
i_eta = eta_solar_lim.numel() // 2
absolute_lim = torch.abs(P_NSI0_num - P_SM_num)
relative_lim = absolute_lim / torch.clamp(torch.abs(P_SM_num), min=REL_FLOOR)

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
colours = ('C0', 'C1', 'C2')
for flavour, colour in zip(range(3), colours):
    axes[0].plot(to_numpy(E_solar_lim), to_numpy(P_SM_num[:, i_eta, flavour]), color=colour, label=f'{FLAVOURS[flavour]} SM')
    axes[0].plot(to_numpy(E_solar_lim), to_numpy(P_NSI0_num[:, i_eta, flavour]), '--', color=colour, label=f'{FLAVOURS[flavour]} NSI-limit')
    axes[1].plot(to_numpy(E_solar_lim), to_numpy(absolute_lim[:, i_eta, flavour]), color=colour, label=FLAVOURS[flavour])
    axes[2].plot(to_numpy(E_solar_lim), to_numpy(relative_lim[:, i_eta, flavour]), color=colour, label=FLAVOURS[flavour])
axes[0].set(ylabel='Probability', title='SM vs NSI-limit'); axes[0].set_ylim(-0.02, 1.02); axes[0].legend(fontsize=7, ncol=2)
axes[1].set(ylabel='Absolute error', title='|Delta P|'); axes[1].set_yscale('log'); axes[1].legend(fontsize=7)
axes[2].set(ylabel='Relative error', title='|Delta P| / max(|P_SM|, 1e-12)'); axes[2].set_yscale('log'); axes[2].legend(fontsize=7)
for ax in axes:
    ax.set_xlabel('E [MeV]'); ax.grid(alpha=0.25)
fig.suptitle(f'Solar to detector, numerical, eta={float(eta_solar_lim[i_eta]):.2f} rad')
fig.tight_layout(); save_and_show('intrinsic2_fig1_solar_limits.png', fig, output_dir=OUTPUT_DIR, show_plots=config.show_plots)

## 4. NSI Presets — Solar Neutrinos to the Detector

Five literature NSI benchmark presets (see `tpeanuts.config.presets.NSI_PRESETS` for the full physics justification and bounds of each) are propagated from solar production to the detector with both the analytical and numerical methods, at fixed $\varepsilon$ for each preset. For each preset, an independent heatmap shows the maximum absolute/relative error over the full (energy, nadir angle) grid, and a line-plot triplet shows $P(\nu_e)$ for both methods (overlaid), the absolute error, and the relative error along an energy scan at fixed nadir angle.

**Expected results:** analytical-numerical agreement should be similar in magnitude to the pure-SM case (Intrinsic Validation 1, Section 4); larger $|\varepsilon|$ presets (e.g. `nsi_lma_dark_esteban2018`, $\varepsilon_{ee}=-2.0$) may show somewhat larger absolute discrepancies since they drive larger matter effects, but relative errors should remain controlled and probability should stay conserved for both methods.

In [ ]:
E_solar_nsi = torch.linspace(0.5, 15.0, 25, device=DEVICE, dtype=DTYPE)
eta_solar_nsi = torch.linspace(0.10, 1.45, 25, device=DEVICE, dtype=DTYPE)
i_eta_nsi = eta_solar_nsi.numel() // 2

nsi_solar_rows = []
for preset_name in NSI_PRESETS:
    epsilon = NSIConfig.from_preset(preset_name).epsilon_tensor(device=DEVICE, real_dtype=DTYPE)
    weights = solar_probability_mass(sm_oscillation, E_solar_nsi, solar, '8B', epsilon=epsilon)
    P_ana = solar_to_detector(weights, sm_oscillation, E_solar_nsi, eta_solar_nsi, 'analytical', epsilon=epsilon)
    P_num = solar_to_detector(weights, sm_oscillation, E_solar_nsi, eta_solar_nsi, 'numerical', epsilon=epsilon)

    row = make_row(preset_name, P_ana, P_num, scenario='Solar to detector', check='analytical vs numerical')
    nsi_solar_rows.append(row)

    heatmap_pair(
        P_ana, P_num, E_solar_nsi, eta_solar_nsi,
        f'NSI preset {preset_name}: solar to detector', 'E [MeV]', 'nadir angle eta [rad]',
        f'intrinsic2_fig2a_nsi_solar_heatmap_{preset_name}.png',
    )
    energy_line_triplet(
        P_ana, P_num, E_solar_nsi, i_eta_nsi, 'analytical', 'numerical',
        f'NSI preset {preset_name}: solar to detector, eta={float(eta_solar_nsi[i_eta_nsi]):.2f} rad',
        f'intrinsic2_fig2b_nsi_solar_energy_{preset_name}.png',
    )

summaries.extend(nsi_solar_rows)
show_table(nsi_solar_rows)
bar_plot(nsi_solar_rows, 'NSI presets: solar to detector, analytical vs numerical', 'intrinsic2_fig2c_nsi_solar_bar.png')

## 5. NSI Presets — Atmospheric Neutrinos to the Detector

The same five NSI presets are compared for atmospheric neutrinos, propagated coherently from production through the atmosphere and Earth to the detector, for an initial $\nu_\mu$ (the flavour most sensitive to matter NSI in atmospheric analyses such as IceCube DeepCore). As in Section 4, each preset gets an independent heatmap plus an energy-scan line triplet (here $P(\nu_e)$ is the $\nu_\mu\to\nu_e$ appearance probability, since the initial flavour is fixed at $\nu_\mu$).

**Expected results:** similar analytical-numerical agreement to the pure-SM atmospheric case; up-going trajectories (long Earth chords) are expected to show the largest discrepancies, consistent with Intrinsic Validation 1's Earth-crossing results.

In [ ]:
E_atm_nsi = torch.logspace(2.0, 5.0, 20, device=DEVICE, dtype=DTYPE)
theta_atm_nsi = torch.linspace(0.0, 180.0, 20, device=DEVICE, dtype=DTYPE)
i_theta_nsi = theta_atm_nsi.numel() // 2
state_numu_3 = torch.eye(3, device=DEVICE, dtype=CDTYPE)[1]

nsi_atm_rows = []
for preset_name in NSI_PRESETS:
    epsilon = NSIConfig.from_preset(preset_name).epsilon_tensor(device=DEVICE, real_dtype=DTYPE)
    P_ana = atmosphere_to_detector(state_numu_3, sm_oscillation, E_atm_nsi, theta_atm_nsi, 'analytical', epsilon=epsilon)
    P_num = atmosphere_to_detector(state_numu_3, sm_oscillation, E_atm_nsi, theta_atm_nsi, 'numerical', epsilon=epsilon)

    row = make_row(preset_name, P_ana, P_num, scenario='Atmosphere to detector (nu_mu)', check='analytical vs numerical')
    nsi_atm_rows.append(row)

    heatmap_pair(
        P_ana, P_num, E_atm_nsi, theta_atm_nsi,
        f'NSI preset {preset_name}: atmosphere to detector (nu_mu)', 'E [MeV]', 'zenith angle [deg]',
        f'intrinsic2_fig3a_nsi_atm_heatmap_{preset_name}.png',
    )
    energy_line_triplet(
        P_ana, P_num, E_atm_nsi, i_theta_nsi, 'analytical', 'numerical',
        f'NSI preset {preset_name}: atmosphere to detector (nu_mu), theta={float(theta_atm_nsi[i_theta_nsi]):.0f} deg',
        f'intrinsic2_fig3b_nsi_atm_energy_{preset_name}.png', log_energy=True,
    )

summaries.extend(nsi_atm_rows)
show_table(nsi_atm_rows)
bar_plot(nsi_atm_rows, 'NSI presets: atmosphere to detector (nu_mu), analytical vs numerical', 'intrinsic2_fig3c_nsi_atm_bar.png')

## 6. Sterile Presets — Solar Neutrinos to the Detector

Three 3+1 sterile presets are compared: the global best fit *(Giunti, Marrone & Palazzo 2017)*, the IceCube atmospheric benchmark, and the two-flavour disappearance-limit construction (see `tpeanuts.config.presets` for the full description of each). Solar production weights use the active 3-flavour sector only (Section 0.4); the resulting 4-component mass weight is propagated through the full $4\times4$ Earth evolutor.

**Expected results:** analytical-numerical agreement at the same order as the SM and NSI cases; the sterile-flavour probability should be small but generally non-zero (unlike Section 3's exact null-mixing limit), reflecting genuine active-sterile conversion in the Earth matter.

In [ ]:
E_solar_sterile = torch.linspace(0.5, 15.0, 25, device=DEVICE, dtype=DTYPE)
eta_solar_sterile = torch.linspace(0.10, 1.45, 25, device=DEVICE, dtype=DTYPE)
i_eta_sterile = eta_solar_sterile.numel() // 2

sterile_solar_rows = []
for preset_name in STERILE_PRESETS:
    oscillation = PropagationConfig.oscillation_parameters_from_preset(preset_name, context=context, antinu=False)
    weights = sterile_solar_weights(oscillation, E_solar_sterile)
    P_ana = solar_to_detector(weights, oscillation, E_solar_sterile, eta_solar_sterile, 'analytical')
    P_num = solar_to_detector(weights, oscillation, E_solar_sterile, eta_solar_sterile, 'numerical')

    row = make_row(preset_name, P_ana, P_num, scenario='Solar to detector', check='analytical vs numerical')
    sterile_solar_rows.append(row)
    print(f"  {preset_name}: max sterile-flavour probability P_s (numerical) = {P_num[..., 3].max().item():.3e}")

    heatmap_pair(
        P_ana, P_num, E_solar_sterile, eta_solar_sterile,
        f'Sterile preset {preset_name}: solar to detector', 'E [MeV]', 'nadir angle eta [rad]',
        f'intrinsic2_fig4a_sterile_solar_heatmap_{preset_name}.png',
    )
    energy_line_triplet(
        P_ana, P_num, E_solar_sterile, i_eta_sterile, 'analytical', 'numerical',
        f'Sterile preset {preset_name}: solar to detector, eta={float(eta_solar_sterile[i_eta_sterile]):.2f} rad',
        f'intrinsic2_fig4b_sterile_solar_energy_{preset_name}.png',
    )

summaries.extend(sterile_solar_rows)
show_table(sterile_solar_rows)
bar_plot(sterile_solar_rows, 'Sterile presets: solar to detector, analytical vs numerical', 'intrinsic2_fig4c_sterile_solar_bar.png')

## 7. Sterile Presets — Atmospheric Neutrinos to the Detector

The same three sterile presets are compared for atmospheric neutrinos, propagated coherently for an initial $\nu_\mu$ through the atmosphere and Earth (the same benchmark IceCube atmospheric analyses target with the $\nu_\mu$ disappearance channel).

**Expected results:** analytical-numerical agreement at the same order as Section 5; `sterile_3p1_benchmark_icecube` (large $\theta_{24}$, $\Delta m^2_{41}=0.3\,{\rm eV}^2$) is expected to show the most visible $\nu_\mu$ disappearance, while both propagation methods should still agree with each other and conserve probability across all four flavours.

In [ ]:
E_atm_sterile = torch.logspace(2.0, 5.0, 20, device=DEVICE, dtype=DTYPE)
theta_atm_sterile = torch.linspace(0.0, 180.0, 20, device=DEVICE, dtype=DTYPE)
i_theta_sterile = theta_atm_sterile.numel() // 2
state_numu_4 = torch.eye(4, device=DEVICE, dtype=CDTYPE)[1]

sterile_atm_rows = []
for preset_name in STERILE_PRESETS:
    oscillation = PropagationConfig.oscillation_parameters_from_preset(preset_name, context=context, antinu=False)
    P_ana = atmosphere_to_detector(state_numu_4, oscillation, E_atm_sterile, theta_atm_sterile, 'analytical')
    P_num = atmosphere_to_detector(state_numu_4, oscillation, E_atm_sterile, theta_atm_sterile, 'numerical')

    row = make_row(preset_name, P_ana, P_num, scenario='Atmosphere to detector (nu_mu)', check='analytical vs numerical')
    sterile_atm_rows.append(row)
    print(f"  {preset_name}: max sterile-flavour probability P_s (numerical) = {P_num[..., 3].max().item():.3e}")

    heatmap_pair(
        P_ana, P_num, E_atm_sterile, theta_atm_sterile,
        f'Sterile preset {preset_name}: atmosphere to detector (nu_mu)', 'E [MeV]', 'zenith angle [deg]',
        f'intrinsic2_fig5a_sterile_atm_heatmap_{preset_name}.png',
    )
    energy_line_triplet(
        P_ana, P_num, E_atm_sterile, i_theta_sterile, 'analytical', 'numerical',
        f'Sterile preset {preset_name}: atmosphere to detector (nu_mu), theta={float(theta_atm_sterile[i_theta_sterile]):.0f} deg',
        f'intrinsic2_fig5b_sterile_atm_energy_{preset_name}.png', log_energy=True,
    )

summaries.extend(sterile_atm_rows)
show_table(sterile_atm_rows)
bar_plot(sterile_atm_rows, 'Sterile presets: atmosphere to detector (nu_mu), analytical vs numerical', 'intrinsic2_fig5c_sterile_atm_bar.png')

## 8. Summary

The table collects every discrepancy and unitarity metric saved by Sections 3-7 (20 rows: 4 SM-vs-limit checks, 5 NSI presets x 2 chains, 3 sterile presets x 2 chains). It is also exported as `validation_intrinsic2_summary.csv`.

**Expected results:** every metric is finite; the four SM-vs-limit rows from Section 3 are several orders of magnitude smaller than the preset-comparison rows from Sections 4-7; both unitarity columns remain below $10^{-6}$ throughout.

In [ ]:
summary = pd.DataFrame(summaries).set_index('case')
numeric_cols = summary.select_dtypes(include='number').columns
display(summary.style.format({c: '{:.3e}' for c in numeric_cols}))
summary.to_csv(OUTPUT_DIR / 'validation_intrinsic2_summary.csv')

assert np.isfinite(summary[numeric_cols].to_numpy()).all(), 'Validation produced non-finite metrics.'
assert summary[['unitarity_1', 'unitarity_2']].to_numpy().max() < 1e-6
print(f'All {len(summary)} BSM intrinsic comparisons completed successfully.')

### Interpretation

Sections 3's four rows isolate pure BSM-formalism correctness (both branches must reduce exactly to the Standard Model at their trivial limit, independently for each propagation method) from Sections 4-7's twenty comparisons, which isolate perturbative/numerical agreement across a representative span of the NSI and sterile parameter space actually explored elsewhere in this repository's physics validation notebooks (`notebooks/validation/physics/BSM/`). A consistently small Section-3 discrepancy together with a consistently small (and unitarity-conserving) Sections 4-7 discrepancy is the strongest available intrinsic evidence that the BSM Hamiltonian construction, the perturbative first-order correction, and the numerical segment reference are all mutually consistent for both the NSI and the 3+1 sterile extensions.